# CodeSplitter

`CodeSplitter`는 프로그래밍 코드를 언어별 특성에 맞게 분할합니다.

## 주요 특징

- **언어별 최적화**: 각 프로그래밍 언어의 문법 구조를 이해하여 분할
- **구문 보존**: 함수, 클래스 등 의미 단위를 유지하면서 분할
- **다양한 언어 지원**: Python, JavaScript, TypeScript, Java, C++, Go 등

## 지원 언어

LangChain은 `Language` enum을 통해 다양한 언어를 지원합니다:
- Python, JavaScript, TypeScript
- Java, C++, C#, Go, Rust
- HTML, CSS, Markdown
- 기타 20+ 언어

## 사용 시나리오

- ✅ **코드 문서화**: 코드를 분할하여 문서 생성
- ✅ **코드 검색 시스템**: RAG 기반 코드 검색
- ✅ **코드 리뷰 자동화**: 코드 분석 및 리뷰


# 05. CodeSplitter — 코드 분할

## 학습 목표
- `RecursiveCharacterTextSplitter.from_language()`로 언어별 코드를 분할해요
- Python, JavaScript, Markdown, HTML의 분할 결과를 확인해요
- 코드 기반 RAG 시스템(코드 검색, 문서화)의 기초를 이해해요

## 사전 지식
- 02-RecursiveCharacterTextSplitter 완료
- 기본 프로그래밍 개념(함수, 클래스) 이해

---

> **왜 코드 전용 분할기가 필요한가?**
>
> 일반 텍스트 분할기는 코드 문법을 모르기 때문에 함수나 클래스 중간에서 끊어버릴 수 있어요. `from_language()`를 사용하면 각 언어의 구문 구조에 맞는 구분자를 자동으로 설정해요.


> 🔑 **핵심 개념**: `from_language()` 메서드는 해당 언어에 최적화된 구분자 목록을 자동으로 설정합니다. Python의 경우 `\nclass `, `\ndef `, `\n\tdef ` 등 코드 구조를 인식하는 구분자가 사용됩니다.

## 지원 언어 목록 확인


In [1]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

# 지원되는 언어의 전체 목록
supported_languages = [e.value for e in Language]
print(f"지원되는 언어 개수: {len(supported_languages)}")
print(f"\n지원 언어 목록:")
for i, lang in enumerate(supported_languages, 1):
    print(f"  {i}. {lang}", end="  ")
    if i % 5 == 0:
        print()


지원되는 언어 개수: 27

지원 언어 목록:
  1. cpp    2. go    3. java    4. kotlin    5. js  
  6. ts    7. php    8. proto    9. python    10. rst  
  11. ruby    12. rust    13. scala    14. swift    15. markdown  
  16. latex    17. html    18. sol    19. csharp    20. cobol  
  21. c    22. lua    23. perl    24. haskell    25. elixir  
  26. powershell    27. visualbasic6  

## 1. Python 코드 분할


> 🎯 **강의 포인트**: 코드 기반 RAG 시스템(사내 코드베이스 검색, 코드 Q&A)을 구축할 때 일반 텍스트 분할기를 쓰면 함수 중간에서 잘릴 수 있습니다. `from_language()`를 사용하면 함수/클래스 단위가 하나의 청크에 담깁니다.

In [2]:
# ============================================================
# TODO: RecursiveCharacterTextSplitter.from_language()로 Python 코드를 분할해보세요
# 힌트: RecursiveCharacterTextSplitter.from_language(language=Language.PYTHON, chunk_size=100, chunk_overlap=0)
# 예상 결과: 클래스와 함수 경계를 인식하여 분할된 11개의 청크가 생성됩니다
# ============================================================

PYTHON_CODE = """
class DataProcessor:
    def __init__(self, data):
        self.data = data
        self.processed = False
    
    def process(self):
        \"\"\"데이터 처리 메서드\"\"\"
        if not self.data:
            raise ValueError("데이터가 비어있습니다")
        
        result = []
        for item in self.data:
            processed_item = self._transform(item)
            result.append(processed_item)
        
        self.processed = True
        return result
    
    def _transform(self, item):
        \"\"\"내부 변환 메서드\"\"\"
        return item.strip().lower()

def main():
    processor = DataProcessor(["Hello", " World ", "Python"])
    result = processor.process()
    print(result)

if __name__ == "__main__":
    main()
"""

# 1단계: Python 언어에 최적화된 Splitter 생성
# - from_language(): 해당 언어의 구문 구조(클래스, 함수, 들여쓰기)에 맞는 구분자를 자동 설정
# - Language.PYTHON: Python 문법 구조 사용
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=100,  # 작은 크기로 설정하여 분할 확인
    chunk_overlap=0
)

python_docs = python_splitter.create_documents([PYTHON_CODE])

print(f"분할된 Python 코드 청크 개수: {len(python_docs)}")
for i, doc in enumerate(python_docs):
    print(f"\n{'=' * 60}")
    print(f"[Chunk {i}]")
    print('=' * 60)
    print(doc.page_content)

분할된 Python 코드 청크 개수: 11

[Chunk 0]
class DataProcessor:
    def __init__(self, data):
        self.data = data

[Chunk 1]
self.processed = False

[Chunk 2]
def process(self):
        """데이터 처리 메서드"""
        if not self.data:

[Chunk 3]
raise ValueError("데이터가 비어있습니다")

[Chunk 4]
result = []
        for item in self.data:

[Chunk 5]
processed_item = self._transform(item)
            result.append(processed_item)

[Chunk 6]
self.processed = True
        return result

[Chunk 7]
def _transform(self, item):
        """내부 변환 메서드"""
        return item.strip().lower()

[Chunk 8]
def main():
    processor = DataProcessor(["Hello", " World ", "Python"])

[Chunk 9]
result = processor.process()
    print(result)

[Chunk 10]
if __name__ == "__main__":
    main()


## 2. JavaScript 코드 분할


In [3]:
JS_CODE = """
class UserManager {
  constructor(database) {
    this.db = database;
    this.users = [];
  }

  async addUser(userData) {
    try {
      const user = await this.db.insert(userData);
      this.users.push(user);
      return user;
    } catch (error) {
      console.error('Failed to add user:', error);
      throw error;
    }
  }

  findUser(id) {
    return this.users.find(user => user.id === id);
  }
}

const manager = new UserManager(database);
manager.addUser({ name: 'John', email: 'john@example.com' });
"""

js_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.JS,
    chunk_size=120,
    chunk_overlap=0
)

js_docs = js_splitter.create_documents([JS_CODE])
print(f"JavaScript 코드 청크 개수: {len(js_docs)}")
print(f"\n첫 번째 청크:")
print(js_docs[0].page_content)


JavaScript 코드 청크 개수: 6

첫 번째 청크:
class UserManager {
  constructor(database) {
    this.db = database;
    this.users = [];
  }


> 💡 **실무 팁**: 코드 청크의 `chunk_size`는 함수 평균 길이를 기준으로 설정하세요. 짧은 유틸리티 함수가 많은 코드베이스라면 `chunk_size=200`, 복잡한 클래스가 많다면 `chunk_size=1000~2000`이 적절합니다.

> ⚠️ **자주 하는 실수**: Markdown 코드 블록(```python ... ```) 안의 코드를 `Language.PYTHON`으로 분할하면 백틱이 포함되어 의도하지 않은 결과가 나옵니다. 코드 파일과 Markdown 파일은 분리해서 처리하세요.

## 3. Markdown 문서 분할


In [4]:
markdown_text = """
# LangChain 시작하기

## 소개

LangChain은 LLM 기반 애플리케이션을 구축하기 위한 프레임워크입니다.

### 주요 기능

- **체인**: 여러 컴포넌트를 연결
- **에이전트**: 자율적으로 작업 수행
- **메모리**: 대화 기록 관리

## 설치

```bash
pip install langchain
```

## 빠른 시작

간단한 예제로 시작해보세요!
"""

md_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.MARKDOWN,
    chunk_size=100,
    chunk_overlap=0
)

md_docs = md_splitter.create_documents([markdown_text])
print(f"Markdown 청크 개수: {len(md_docs)}")
for i, doc in enumerate(md_docs):
    print(f"\n[Chunk {i}]")
    print(doc.page_content)


Markdown 청크 개수: 3

[Chunk 0]
# LangChain 시작하기

## 소개

LangChain은 LLM 기반 애플리케이션을 구축하기 위한 프레임워크입니다.

[Chunk 1]
### 주요 기능

- **체인**: 여러 컴포넌트를 연결
- **에이전트**: 자율적으로 작업 수행
- **메모리**: 대화 기록 관리

[Chunk 2]
## 설치

```bash
pip install langchain
```

## 빠른 시작

간단한 예제로 시작해보세요!


## 핵심 정리 및 마무리

### 지원 언어 (27가지)

Python, JavaScript, TypeScript, Java, C++, C#, Go, Rust, HTML, CSS, Markdown, LaTeX, Ruby, Scala, Swift, Kotlin, PHP, Haskell, Lua, Perl, Elixir 등

### 활용 시나리오
> 코드 검색 RAG 시스템을 구축할 때 함수 단위로 분할하면 "이 기능을 어떻게 구현하나요?"라는 질문에 정확한 코드 조각을 반환할 수 있어요. `chunk_size`를 함수 평균 길이로 설정하는 게 좋아요.

---

## 다음 예고

특수 형식(구조화된 문서) 분할 방법을 이어서 배워볼게요.

- **06-MarkdownHeaderTextSplitter**: 헤더 기반으로 문서 계층 구조를 보존
- **07-HTMLHeaderTextSplitter**: HTML 태그 기반 웹 페이지 분할
- **08-RecursiveJsonSplitter**: JSON 구조를 유지하면서 분할
